In [0]:
"""
Bronze layer ingest — downloads REAL Binance kline data (not synthetic).

- Monthly file  -> FULL LOAD        -> saved as {SYMBOL}-{INTERVAL}-{YYYY}-{MM}.csv
- Daily file    -> INCREMENTAL LOAD -> saved as {SYMBOL}-{INTERVAL}-{YYYY}-{MM}-{DD}.csv

Every download is checksum-verified (SHA256) against Binance's own .CHECKSUM
sidecar file before being trusted and written to disk.

Run this from a Databricks notebook cell (01_bronze_ingest.py) or as a
standalone script.
"""

import hashlib
import io
import zipfile
from pathlib import Path
from datetime import date, timedelta

import requests

# --------------------------------------------------------------------------
# CONFIG — tune these to hit the size targets from the review
# --------------------------------------------------------------------------
INTERVAL = "1m"                       # 1-minute candles (not 1h) -> much larger files
FULL_LOAD_YEAR = 2026
FULL_LOAD_MONTH = 8                   # month used for the full (monthly) load
INCR_LOAD_DATE = date(2026, 9, 25)    # day used for the incremental (daily) load

# --------------------------------------------------------------------------
# Output paths — running locally in VSCode, so files just land in folders
# next to this script (no Databricks paths needed here; you'll upload the
# resulting CSVs into your Databricks Volume afterward).
# --------------------------------------------------------------------------
SCRIPT_DIR = Path(__file__).resolve().parent

FULL_LOAD_DIR = SCRIPT_DIR / "full_load"
INCR_LOAD_DIR = SCRIPT_DIR / "incremental_load"
FULL_LOAD_DIR.mkdir(parents=True, exist_ok=True)
INCR_LOAD_DIR.mkdir(parents=True, exist_ok=True)

BASE_URL = "https://data.binance.vision/data/spot"
TICKER_24H_URL = "https://api.binance.com/api/v3/ticker/24hr"

REQUEST_TIMEOUT = 30


# --------------------------------------------------------------------------
# Symbol universe — top N USDT pairs by 24h quote volume (fetched live)
# --------------------------------------------------------------------------
def get_top_usdt_symbols(n: int = 50) -> list[str]:
    """Return the top-N USDT spot pairs ranked by 24h quote volume."""
    resp = requests.get(TICKER_24H_URL, timeout=REQUEST_TIMEOUT)
    resp.raise_for_status()
    tickers = resp.json()

    usdt_pairs = [t for t in tickers if t["symbol"].endswith("USDT")]
    usdt_pairs.sort(key=lambda t: float(t["quoteVolume"]), reverse=True)

    return [t["symbol"] for t in usdt_pairs[:n]]


# --------------------------------------------------------------------------
# Core download + checksum-verify + extract helper
# --------------------------------------------------------------------------
def _download_verify_extract(zip_url: str, target_csv_name: str, output_dir: Path) -> Path:
    """
    Download <zip_url>, verify it against <zip_url>.CHECKSUM, extract the
    single CSV inside, and save it to output_dir/<target_csv_name>.
    Returns the path to the saved CSV, or raises on failure.
    """
    zip_resp = requests.get(zip_url, timeout=REQUEST_TIMEOUT)
    if zip_resp.status_code == 404:
        raise FileNotFoundError(f"No data at {zip_url} (symbol may not exist for this date)")
    zip_resp.raise_for_status()

    checksum_resp = requests.get(zip_url + ".CHECKSUM", timeout=REQUEST_TIMEOUT)
    checksum_resp.raise_for_status()
    expected_sha256 = checksum_resp.text.split()[0].strip()

    actual_sha256 = hashlib.sha256(zip_resp.content).hexdigest()
    if actual_sha256 != expected_sha256:
        raise ValueError(
            f"Checksum mismatch for {zip_url}\n"
            f"  expected: {expected_sha256}\n"
            f"  actual:   {actual_sha256}"
        )

    with zipfile.ZipFile(io.BytesIO(zip_resp.content)) as z:
        inner_name = z.namelist()[0]
        raw_bytes = z.read(inner_name)

    out_path = output_dir / target_csv_name
    out_path.write_bytes(raw_bytes)
    print(f"  [OK] checksum verified -> {out_path} ({len(raw_bytes)/1_048_576:.2f} MB)")
    return out_path


# --------------------------------------------------------------------------
# Monthly (FULL LOAD) and Daily (INCREMENTAL LOAD) downloaders
# --------------------------------------------------------------------------
def download_monthly_full_load(symbol: str, interval: str, year: int, month: int) -> Path:
    """Full load: one monthly klines file, verified and saved as
    {SYMBOL}-{INTERVAL}-{YYYY}-{MM}.csv"""
    remote_stub = f"{symbol}-{interval}-{year}-{month:02d}"
    zip_url = f"{BASE_URL}/monthly/klines/{symbol}/{interval}/{remote_stub}.zip"
    target_name = f"{remote_stub}.csv"
    return _download_verify_extract(zip_url, target_name, FULL_LOAD_DIR)


def download_daily_incremental_load(symbol: str, interval: str, day: date) -> Path:
    """Incremental load: one daily klines file, verified and saved as
    {SYMBOL}-{INTERVAL}-{YYYY}-{MM}-{DD}.csv"""
    remote_stub = f"{symbol}-{interval}-{day.year}-{day.month:02d}-{day.day:02d}"
    zip_url = f"{BASE_URL}/daily/klines/{symbol}/{interval}/{remote_stub}.zip"
    target_name = f"{remote_stub}.csv"
    return _download_verify_extract(zip_url, target_name, INCR_LOAD_DIR)


# --------------------------------------------------------------------------
# Main — loop over the whole symbol universe for both loads
# --------------------------------------------------------------------------
def main():
    symbols = get_top_usdt_symbols(n=50)
    print(f"Tracking {len(symbols)} USDT pairs: {symbols[:5]} ...")

    print("\n=== FULL LOAD (monthly files) ===")
    full_load_files, full_load_failures = [], []
    for symbol in symbols:
        try:
            print(f"[{symbol}] downloading monthly {INTERVAL} klines for {FULL_LOAD_YEAR}-{FULL_LOAD_MONTH:02d}")
            path = download_monthly_full_load(symbol, INTERVAL, FULL_LOAD_YEAR, FULL_LOAD_MONTH)
            full_load_files.append(path)
        except Exception as e:
            print(f"  [SKIP] {symbol}: {e}")
            full_load_failures.append(symbol)

    print("\n=== INCREMENTAL LOAD (daily file) ===")
    incr_load_files, incr_load_failures = [], []
    for symbol in symbols:
        try:
            print(f"[{symbol}] downloading daily {INTERVAL} klines for {INCR_LOAD_DATE.isoformat()}")
            path = download_daily_incremental_load(symbol, INTERVAL, INCR_LOAD_DATE)
            incr_load_files.append(path)
        except Exception as e:
            print(f"  [SKIP] {symbol}: {e}")
            incr_load_failures.append(symbol)

    total_full_mb = sum(f.stat().st_size for f in full_load_files) / 1_048_576
    total_incr_mb = sum(f.stat().st_size for f in incr_load_files) / 1_048_576

    print("\n=== SUMMARY ===")
    print(f"Full load:        {len(full_load_files)} files, {total_full_mb:.1f} MB total  -> {FULL_LOAD_DIR}  (failed: {full_load_failures})")
    print(f"Incremental load: {len(incr_load_files)} files, {total_incr_mb:.1f} MB total  -> {INCR_LOAD_DIR}  (failed: {incr_load_failures})")


if __name__ == "__main__":
    main()